In [2]:
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import TimeSliderChoropleth
import pyarrow.parquet as pq
import os
import json
from io import StringIO
import branca.colormap as cm
import numpy as np

# ==============================================================================
# EENVOUDIGERE ANIMATIE MET TIMESLIDERCHOROPLETH
# ==============================================================================

# --- Configuratie ---
TOMTOM_FILE_PATH = os.path.join('Data', '20250820163000_stream.tomtom.analyze-sail.parquet')
SHAPEFILE_PATH = r'C:\Users\gerri\Downloads\01-12-2024\01-12-2024\Wegvakken\Wegvakken.shp'
# We behouden 50% sample voor een balans tussen data en snelheid
SAMPLE_FRACTION = 0.1
SHAPEFILE_ID_COL = 'WVK_ID'


def parse_tomtom_data(row):
    """Verwerkt de data uit de '_value' kolom en voegt de timestamp toe."""
    value_string = row['_value']
    try:
        start_index = value_string.find('{')
        if start_index == -1: return None
        
        json_data = json.loads(value_string[start_index:])
        # Converteer tijd naar UTC en verwijder timezone info voor compatibiliteit
        timestamp = pd.to_datetime(json_data.get('time')).tz_convert(None)
        
        if 'data' in json_data and isinstance(json_data['data'], str):
            csv_data = pd.read_csv(StringIO(json_data['data']))
            if not csv_data.empty:
                csv_data['time'] = timestamp # Voeg de timestamp toe aan elke rij
                return csv_data
    except (json.JSONDecodeError, KeyError, ValueError):
        return None
    return None

def create_animated_traffic_map_simple():
    """Voert het volledige proces uit met TimeSliderChoropleth."""
    try:
        # --- 1. Data Laden, Verwerken en Aggregeren per Minuut ---
        print(f"Laden van {SAMPLE_FRACTION*100}% sample uit {TOMTOM_FILE_PATH}...")
        df_raw = pd.read_parquet(TOMTOM_FILE_PATH).sample(frac=SAMPLE_FRACTION, random_state=1)
        
        print("Verwerken van verkeersdata inclusief timestamps...")
        parsed_dfs = df_raw.apply(parse_tomtom_data, axis=1)
        df_traffic = pd.concat(parsed_dfs.dropna().tolist(), ignore_index=True)
        
        print("Aggregeren van data per wegvak per minuut...")
        df_traffic['time_bin'] = df_traffic['time'].dt.floor('T') # 'T' = minuut
        df_agg = df_traffic.groupby(['id', 'time_bin'])['traffic_level'].mean().reset_index()
        print(f"Data geaggregeerd naar {len(df_agg):,} unieke tijd-wegvak combinaties.")

        # --- 2. Kaart Laden en Voorbereiden ---
        print(f"\nLaden van kaartdata: {SHAPEFILE_PATH}...")
        gdf = gpd.read_file(SHAPEFILE_PATH)
        gdf = gdf.to_crs(epsg=4326)
        
        print("Filteren op Amsterdam...")
        min_lon, min_lat, max_lon, max_lat = 4.72, 52.28, 5.08, 52.43
        gdf_amsterdam = gdf.cx[min_lon:max_lon, min_lat:max_lat].copy()
        
        print("Agressief vereenvoudigen van geometrie...")
        gdf_amsterdam['geometry'] = gdf_amsterdam.geometry.simplify(tolerance=0.0001, preserve_topology=True)

        # --- 3. Data Koppelen ---
        print("\nKoppelen van data...")
        gdf_amsterdam[SHAPEFILE_ID_COL] = pd.to_numeric(gdf_amsterdam[SHAPEFILE_ID_COL], errors='coerce').astype('Int64').astype(str)
        df_agg['id'] = pd.to_numeric(df_agg['id'], errors='coerce').astype('Int64').astype(str)
        
        # We gebruiken nu een 'left' merge om ALLE wegvakken te behouden, zelfs als ze geen data hebben voor een tijdstap
        merged_gdf = gdf_amsterdam.merge(df_agg, left_on=SHAPEFILE_ID_COL, right_on='id', how='left')
        print(f"Succesvol {len(merged_gdf):,} rijen voorbereid voor animatie.")

        # --- Belangrijk: Opschonen van Datumkolommen ---
        print("Opschonen van datumkolommen...")
        for col in merged_gdf.columns:
            if pd.api.types.is_datetime64_any_dtype(merged_gdf[col]):
                merged_gdf[col] = merged_gdf[col].astype(str)
                
        # --- 4. Data Structureren voor TimeSliderChoropleth ---
        print("Structureren van data voor de tijdslider...")
        
        # Converteer tijd-bin naar Unix timestamp (in milliseconden, als integer)
        merged_gdf['time_unix'] = pd.to_datetime(merged_gdf['time_bin']).astype(np.int64) // 10**6
        
        # Maak de colormap
        colormap = cm.LinearColormap(colors=['green', 'yellow', 'red'], vmin=0, vmax=1)
        
        # Bouw de 'styledict' structuur
        styledict = {}
        all_times = sorted(merged_gdf['time_unix'].unique())

        for index, row in merged_gdf.iterrows():
            wvk_id = str(row[SHAPEFILE_ID_COL]) # Zorg ervoor dat ID een string is
            time_ms = row['time_unix']
            level = row['traffic_level']
            
            if wvk_id not in styledict:
                styledict[wvk_id] = {}
            
            # Bepaal de kleur, of grijs als er geen data is
            color = '#d3d3d3' # Grijs als standaard/geen data
            if not pd.isna(level):
                color = colormap(level)
                
            styledict[wvk_id][time_ms] = {
                'color': color,
                'opacity': 0.8 if not pd.isna(level) else 0.1 # Maak vakken zonder data bijna transparant
            }
            
        # Vul ontbrekende tijdstappen op met de standaard grijze stijl
        for wvk_id in styledict:
            for time_ms in all_times:
                if time_ms not in styledict[wvk_id]:
                    styledict[wvk_id][time_ms] = {'color': '#d3d3d3', 'opacity': 0.1}


        # --- 5. Visualisatie met TimeSliderChoropleth ---
        print("Genereren van de geanimeerde Folium verkeerskaart...")
        
        amsterdam_location = [52.3676, 4.9041]
        m = folium.Map(location=amsterdam_location, zoom_start=12, tiles="CartoDB positron")

        TimeSliderChoropleth(
            data=merged_gdf.to_json(), # Geef de volledige GeoJSON door
            styledict=styledict,
        ).add_to(m)

        m.add_child(colormap) # Voeg de legenda toe
        print("\n--- SUCCES! ---")
        
        m.save("verkeerskaart_amsterdam_timeslider.html")
        print("Een interactieve, geanimeerde kaart is opgeslagen als 'verkeerskaart_amsterdam_timeslider.html'")
        
        return m

    except Exception as e:
        print(f"Er is een onverwachte fout opgetreden: {e}")
        return None

# Voer het script uit en toon de kaart
traffic_map = create_animated_traffic_map_simple()
traffic_map



AttributeError: partially initialized module 'pandas' has no attribute '_pandas_datetime_CAPI' (most likely due to a circular import)